In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = r"C:\Users\hp\OneDrive\Desktop\celebal-Technologies\week6\pyspark\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\hp\OneDrive\Desktop\celebal-Technologies\week6\pyspark\Scripts\python.exe"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

print("Done!")

Done!


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("CelebalAssignment") \
    .master("local[2]") \
    .config("spark.python.worker.reuse", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Started! Version:", spark.version)

Spark Started! Version: 4.1.2


In [4]:
raw_data = [
    (1,  "Alice",   29, "Electronics", "North", 1500.50),
    (2,  "Bob",     35, "Clothing",    "South", 800.00),
    (3,  "Charlie", 17, "Electronics", "East",  200.00),
    (4,  "Diana",   42, "Furniture",   "West",  3200.75),
    (5,  "Eve",    None, "Clothing",   "North", 450.00),
    (6,  "Frank",   28,  None,         "South", 600.00),
    (7,  "Grace",   33, "Electronics", "East",  None),
    (8,  "Hank",    55, "Furniture",    None,   1800.00),
    (9,  "Ivy",     23, "Clothing",    "West",  320.00),
    (10, "Jack",    31, "Electronics", "North", 2100.00),
    (2,  "Bob",     35, "Clothing",    "South", 800.00),
    (11, "Karen",   45, "Furniture",   "East",  4500.00),
    (12, "Leo",     19, "Electronics", "South", 950.00),
    (13, "Mona",    38, "Clothing",    "West",  670.00),
    (14, "Nate",    62, "Furniture",   "North", 2800.00),
    (15, "Olivia",  27, "Electronics", "East",  1350.00),
    (1,  "Alice",   29, "Electronics", "North", 1500.50),
    (16, "Paul",    34, "Clothing",    "South", 530.00),
    (17, "Quinn",  None, "Furniture",  "West",  2200.00),
    (18, "Rose",    41, "Electronics", "North", 1750.00),
    (19, "Sam",     26, "Clothing",    "East",  410.00),
    (20, "Tina",    50, "Furniture",   "South", 3800.00),
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name",        StringType(),  True),
    StructField("age",         IntegerType(), True),
    StructField("category",    StringType(),  True),
    StructField("region",      StringType(),  True),
    StructField("amount",      DoubleType(),  True),
])

raw_df = spark.createDataFrame(raw_data, schema=schema)
print(f"Total rows: {raw_df.count()}")
raw_df.show()

Total rows: 22
+-----------+-------+----+-----------+------+-------+
|customer_id|   name| age|   category|region| amount|
+-----------+-------+----+-----------+------+-------+
|          1|  Alice|  29|Electronics| North| 1500.5|
|          2|    Bob|  35|   Clothing| South|  800.0|
|          3|Charlie|  17|Electronics|  East|  200.0|
|          4|  Diana|  42|  Furniture|  West|3200.75|
|          5|    Eve|NULL|   Clothing| North|  450.0|
|          6|  Frank|  28|       NULL| South|  600.0|
|          7|  Grace|  33|Electronics|  East|   NULL|
|          8|   Hank|  55|  Furniture|  NULL| 1800.0|
|          9|    Ivy|  23|   Clothing|  West|  320.0|
|         10|   Jack|  31|Electronics| North| 2100.0|
|          2|    Bob|  35|   Clothing| South|  800.0|
|         11|  Karen|  45|  Furniture|  East| 4500.0|
|         12|    Leo|  19|Electronics| South|  950.0|
|         13|   Mona|  38|   Clothing|  West|  670.0|
|         14|   Nate|  62|  Furniture| North| 2800.0|
|         15|

In [5]:
before = raw_df.count()
deduped_df = raw_df.dropDuplicates()
after = deduped_df.count()

print(f"Before: {before} rows")
print(f"After:  {after} rows")
print(f"Removed: {before - after} duplicates")
deduped_df.show()

Before: 22 rows
After:  20 rows
Removed: 2 duplicates
+-----------+-------+----+-----------+------+-------+
|customer_id|   name| age|   category|region| amount|
+-----------+-------+----+-----------+------+-------+
|          4|  Diana|  42|  Furniture|  West|3200.75|
|          2|    Bob|  35|   Clothing| South|  800.0|
|         10|   Jack|  31|Electronics| North| 2100.0|
|          1|  Alice|  29|Electronics| North| 1500.5|
|          3|Charlie|  17|Electronics|  East|  200.0|
|          9|    Ivy|  23|   Clothing|  West|  320.0|
|         18|   Rose|  41|Electronics| North| 1750.0|
|         14|   Nate|  62|  Furniture| North| 2800.0|
|         13|   Mona|  38|   Clothing|  West|  670.0|
|         16|   Paul|  34|   Clothing| South|  530.0|
|         11|  Karen|  45|  Furniture|  East| 4500.0|
|         20|   Tina|  50|  Furniture| South| 3800.0|
|         12|    Leo|  19|Electronics| South|  950.0|
|         19|    Sam|  26|   Clothing|  East|  410.0|
|         15| Olivia|  27|El

In [6]:
from pyspark.sql.functions import col, count, when

print("Null counts:")
deduped_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in deduped_df.columns
]).show()

median_age = int(deduped_df.approxQuantile("age", [0.5], 0.01)[0])
print(f"Median age: {median_age}")

cleaned_df = (deduped_df
    .fillna({"age": median_age, "category": "Unknown", "region": "Unknown"})
    .filter(col("amount").isNotNull())
)

print(f"Clean rows: {cleaned_df.count()}")
cleaned_df.show()

Null counts:
+-----------+----+---+--------+------+------+
|customer_id|name|age|category|region|amount|
+-----------+----+---+--------+------+------+
|          0|   0|  2|       1|     1|     1|
+-----------+----+---+--------+------+------+

Median age: 33
Clean rows: 19
+-----------+-------+---+-----------+-------+-------+
|customer_id|   name|age|   category| region| amount|
+-----------+-------+---+-----------+-------+-------+
|          4|  Diana| 42|  Furniture|   West|3200.75|
|          2|    Bob| 35|   Clothing|  South|  800.0|
|         10|   Jack| 31|Electronics|  North| 2100.0|
|          1|  Alice| 29|Electronics|  North| 1500.5|
|          3|Charlie| 17|Electronics|   East|  200.0|
|          9|    Ivy| 23|   Clothing|   West|  320.0|
|         18|   Rose| 41|Electronics|  North| 1750.0|
|         14|   Nate| 62|  Furniture|  North| 2800.0|
|         13|   Mona| 38|   Clothing|   West|  670.0|
|         16|   Paul| 34|   Clothing|  South|  530.0|
|         11|  Karen| 45

In [7]:
transformed_df = (cleaned_df
    .withColumnRenamed("amount", "sale_amount")
    .withColumn("sale_amount", F.col("sale_amount").cast(DoubleType()))
    .withColumn("amount_inr",  F.round(F.col("sale_amount") * 83.5, 2))
    .withColumn("age_group",
        F.when(F.col("age") < 25, "Youth")
         .when(F.col("age") < 40, "Adult")
         .when(F.col("age") < 60, "Middle-Aged")
         .otherwise("Senior"))
)

transformed_df.printSchema()
transformed_df.show()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = false)
 |-- category: string (nullable = false)
 |-- region: string (nullable = false)
 |-- sale_amount: double (nullable = true)
 |-- amount_inr: double (nullable = true)
 |-- age_group: string (nullable = false)

+-----------+-------+---+-----------+-------+-----------+----------+-----------+
|customer_id|   name|age|   category| region|sale_amount|amount_inr|  age_group|
+-----------+-------+---+-----------+-------+-----------+----------+-----------+
|          4|  Diana| 42|  Furniture|   West|    3200.75| 267262.63|Middle-Aged|
|          2|    Bob| 35|   Clothing|  South|      800.0|   66800.0|      Adult|
|         10|   Jack| 31|Electronics|  North|     2100.0|  175350.0|      Adult|
|          1|  Alice| 29|Electronics|  North|     1500.5| 125291.75|      Adult|
|          3|Charlie| 17|Electronics|   East|      200.0|   16700.0|      Youth|
|          9|    Ivy| 

In [8]:
# Filter 1: Age 18-60
print("Age 18-60:")
transformed_df.filter(
    (F.col("age") >= 18) & (F.col("age") <= 60)
).show()

# Filter 2: Sirf Electronics
print("Electronics only:")
transformed_df.filter(F.col("category") == "Electronics").show()

# Filter 3: North ya East region
print("North/East region:")
transformed_df.filter(F.col("region").isin("North", "East")).show()

# Filter 4: High value sales
print("Sale > 1000:")
transformed_df.filter(F.col("sale_amount") > 1000).show()

Age 18-60:
+-----------+------+---+-----------+-------+-----------+----------+-----------+
|customer_id|  name|age|   category| region|sale_amount|amount_inr|  age_group|
+-----------+------+---+-----------+-------+-----------+----------+-----------+
|          4| Diana| 42|  Furniture|   West|    3200.75| 267262.63|Middle-Aged|
|          2|   Bob| 35|   Clothing|  South|      800.0|   66800.0|      Adult|
|         10|  Jack| 31|Electronics|  North|     2100.0|  175350.0|      Adult|
|          1| Alice| 29|Electronics|  North|     1500.5| 125291.75|      Adult|
|          9|   Ivy| 23|   Clothing|   West|      320.0|   26720.0|      Youth|
|         18|  Rose| 41|Electronics|  North|     1750.0|  146125.0|Middle-Aged|
|         13|  Mona| 38|   Clothing|   West|      670.0|   55945.0|      Adult|
|         16|  Paul| 34|   Clothing|  South|      530.0|   44255.0|      Adult|
|         11| Karen| 45|  Furniture|   East|     4500.0|  375750.0|Middle-Aged|
|         20|  Tina| 50|  Fur

In [9]:
transformed_df.agg(
    F.count("customer_id").alias("total_customers"),
    F.round(F.sum("sale_amount"), 2).alias("total_sales"),
    F.round(F.avg("sale_amount"), 2).alias("avg_sale"),
    F.min("sale_amount").alias("min_sale"),
    F.max("sale_amount").alias("max_sale"),
).show()

+---------------+-----------+--------+--------+--------+
|total_customers|total_sales|avg_sale|min_sale|max_sale|
+---------------+-----------+--------+--------+--------+
|             19|   29931.25| 1575.33|   200.0|  4500.0|
+---------------+-----------+--------+--------+--------+



In [10]:

print("By Category:")
transformed_df.groupBy("category").agg(
    F.count("customer_id").alias("customers"),
    F.round(F.sum("sale_amount"), 2).alias("total_sales"),
    F.round(F.avg("sale_amount"), 2).alias("avg_sale"),
).orderBy(F.desc("total_sales")).show()


print("By Region:")
transformed_df.groupBy("region").agg(
    F.count("customer_id").alias("customers"),
    F.round(F.sum("sale_amount"), 2).alias("total_sales"),
).orderBy(F.desc("total_sales")).show()

By Category:
+-----------+---------+-----------+--------+
|   category|customers|total_sales|avg_sale|
+-----------+---------+-----------+--------+
|  Furniture|        6|   18300.75| 3050.13|
|Electronics|        6|     7850.5| 1308.42|
|   Clothing|        6|     3180.0|   530.0|
|    Unknown|        1|      600.0|   600.0|
+-----------+---------+-----------+--------+

By Region:
+-------+---------+-----------+
| region|customers|total_sales|
+-------+---------+-----------+
|  North|        5|     8600.5|
|  South|        5|     6680.0|
|   East|        4|     6460.0|
|   West|        4|    6390.75|
|Unknown|        1|     1800.0|
+-------+---------+-----------+



In [11]:
print("Categories jahan total sales > 3000:")
transformed_df.groupBy("category") \
    .agg(F.round(F.sum("sale_amount"), 2).alias("total_sales")) \
    .filter(F.col("total_sales") > 3000) \
    .orderBy(F.desc("total_sales")) \
    .show()

Categories jahan total sales > 3000:
+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|  Furniture|   18300.75|
|Electronics|     7850.5|
|   Clothing|     3180.0|
+-----------+-----------+



In [12]:
transformed_df.createOrReplaceTempView("sales")

spark.sql("""
    SELECT 
        category,
        region,
        COUNT(customer_id)         AS customers,
        ROUND(SUM(sale_amount), 2) AS total_sales,
        ROUND(AVG(sale_amount), 2) AS avg_sale
    FROM sales
    WHERE age BETWEEN 18 AND 60
    GROUP BY category, region
    HAVING SUM(sale_amount) > 500
    ORDER BY total_sales DESC
""").show()

+-----------+-------+---------+-----------+--------+
|   category| region|customers|total_sales|avg_sale|
+-----------+-------+---------+-----------+--------+
|  Furniture|   West|        2|    5400.75| 2700.38|
|Electronics|  North|        3|     5350.5|  1783.5|
|  Furniture|   East|        1|     4500.0|  4500.0|
|  Furniture|  South|        1|     3800.0|  3800.0|
|  Furniture|Unknown|        1|     1800.0|  1800.0|
|Electronics|   East|        1|     1350.0|  1350.0|
|   Clothing|  South|        2|     1330.0|   665.0|
|   Clothing|   West|        2|      990.0|   495.0|
|Electronics|  South|        1|      950.0|   950.0|
|    Unknown|  South|        1|      600.0|   600.0|
+-----------+-------+---------+-----------+--------+



In [13]:
final_df = (
    raw_df
    .dropDuplicates()
    .fillna({"age": 33, "category": "Unknown", "region": "Unknown"})
    .filter(F.col("amount").isNotNull())
    .withColumnRenamed("amount", "sale_amount")
    .withColumn("amount_inr", F.round(F.col("sale_amount") * 83.5, 2))
    .withColumn("age_group",
        F.when(F.col("age") < 25, "Youth")
         .when(F.col("age") < 40, "Adult")
         .when(F.col("age") < 60, "Middle-Aged")
         .otherwise("Senior"))
    .filter((F.col("age") >= 18) & (F.col("age") <= 65))
    .cache()
)

print(f"Final rows: {final_df.count()}")
final_df.show()

Final rows: 18
+-----------+------+---+-----------+-------+-----------+----------+-----------+
|customer_id|  name|age|   category| region|sale_amount|amount_inr|  age_group|
+-----------+------+---+-----------+-------+-----------+----------+-----------+
|          6| Frank| 28|    Unknown|  South|      600.0|   50100.0|      Adult|
|         18|  Rose| 41|Electronics|  North|     1750.0|  146125.0|Middle-Aged|
|          4| Diana| 42|  Furniture|   West|    3200.75| 267262.63|Middle-Aged|
|         14|  Nate| 62|  Furniture|  North|     2800.0|  233800.0|     Senior|
|         13|  Mona| 38|   Clothing|   West|      670.0|   55945.0|      Adult|
|          2|   Bob| 35|   Clothing|  South|      800.0|   66800.0|      Adult|
|         10|  Jack| 31|Electronics|  North|     2100.0|  175350.0|      Adult|
|         16|  Paul| 34|   Clothing|  South|      530.0|   44255.0|      Adult|
|          8|  Hank| 55|  Furniture|Unknown|     1800.0|  150300.0|Middle-Aged|
|         11| Karen| 45| 

In [17]:
import subprocess
subprocess.check_call(["pip", "install", "pandas"])
print("Pandas installed!")

Pandas installed!


In [18]:
import os
import pandas as pd

output_path = r"C:\Users\hp\OneDrive\Desktop\celebal-Technologies\week6\output"
os.makedirs(output_path, exist_ok=True)

# CSV save karo
transformed_df.toPandas().to_csv(
    os.path.join(output_path, "sales_data.csv"),
    index=False
)
print(f" CSV saved!")

# CSV wapas read karo
csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"file:///{output_path}/sales_data.csv".replace("\\", "/"))

print(" CSV read successfully!")
csv_df.printSchema()
csv_df.show(5)

✅ CSV saved!
✅ CSV read successfully!
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- amount_inr: double (nullable = true)
 |-- age_group: string (nullable = true)

+-----------+-------+---+-----------+------+-----------+----------+-----------+
|customer_id|   name|age|   category|region|sale_amount|amount_inr|  age_group|
+-----------+-------+---+-----------+------+-----------+----------+-----------+
|          4|  Diana| 42|  Furniture|  West|    3200.75| 267262.63|Middle-Aged|
|          2|    Bob| 35|   Clothing| South|      800.0|   66800.0|      Adult|
|         10|   Jack| 31|Electronics| North|     2100.0|  175350.0|      Adult|
|          1|  Alice| 29|Electronics| North|     1500.5| 125291.75|      Adult|
|          3|Charlie| 17|Electronics|  East|      200.0|   16700.0|      You

In [22]:
spark.stop()
print("All tasks completed!")

All tasks completed!
